# Laboratorio 1 – Exploración, preparación y regresión lineal

**Curso:** ISIS-2611 – Aprendizaje de Máquina  

**Caso:** AlpesPlanck  

**Autores:** Lucas Valbuena 202311538 y Juan Goyeneche 202320863

**Semestre:** 2026-02


## 0. Configuración del entorno


### 0.1 Importación de librerías 

*Se importaron las librerías necesarias para manipulación, visualización y modelado.*

In [353]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

### 0.2 Configuración global de consideraciones


In [354]:
random_state = 42
np.random.seed(random_state)

test_size = 0.25

### 0.3 Carga de los datos


*Se cargaron los datos de entrenamiento desde la carpeta `data/`. El archivo de test se cargará posteriormente, cuando se haya seleccionado el mejor modelo.*

In [355]:
datos_entrenamiento = pd.read_csv("data/Datos Lab 1.csv")
data = datos_entrenamiento.copy()

## 1. Entendimiento del negocio y de los datos


### 1.1 Descripción del caso y objetivo del modelo


*El Instituto AlpesPlanck de Biogeoquímica registra variables meteorológicas en su estación de Jena, Alemania. El objetivo del laboratorio fue explorar estos datos y preparar un modelo de regresión lineal para predecir la temperatura máxima del día siguiente, de manera que la información pudiera apoyar la prevención de riesgos como incendios forestales y los efectos asociados con El Niño y La Niña.*

### 1.2 Revisión del diccionario de datos


In [356]:
diccionario_datos = pd.read_excel("data/Diccionario de datos.xlsx")
display(diccionario_datos)

,variable,tipo,descripcion
0,fecha,texto,"Fecha de la observación, en formato día.mes.año."
1,presion_media,numérico,Presión atmosférica media del día (mbar).
2,presion_min,numérico,Presión atmosférica mínima del día (mbar).
3,presion_max,numérico,Presión atmosférica máxima del día (mbar).
4,presion_desv,numérico,Desviación típica de la presión durante el día...
5,humedad_media,numérico,Humedad relativa media del día (%).
6,humedad_min,numérico,Humedad relativa mínima del día (%).
7,humedad_max,numérico,Humedad relativa máxima del día (%).
8,humedad_desv,numérico,Desviación típica de la humedad relativa (%).
9,viento_media,numérico,Velocidad media del viento durante el día (m/s).


*Mediante la función `display` se mostró el diccionario de variables. Luego, se procedió a categorizar las variables según sus tipos: categóricas, numéricas continuas y numéricas discretas.*

In [357]:
variables_categoricas = diccionario_datos.loc[
    diccionario_datos["tipo"] == "texto", "variable"
].tolist()

variables_numericas = diccionario_datos.loc[
    diccionario_datos["tipo"] == "numérico", "variable"
].tolist()
variables_numericas_discretas = [
    "registros_del_dia",
    "anio",
    "dia_del_anio",
]
variables_numericas_continuas = [
    variable
    for variable in variables_numericas
    if variable not in variables_numericas_discretas
]

print(f"Variables categóricas: {len(variables_categoricas)}")
print(f"Variables numéricas continuas: {len(variables_numericas_continuas)}")
print(f"Variables numéricas discretas: {len(variables_numericas_discretas)}")

Variables categóricas: 4
Variables numéricas continuas: 20
Variables numéricas discretas: 3


### 1.3 Definición de la variable objetivo (temperatura máxima del día)


*Se definió explícitamente la temperatura máxima del día como variable objetivo.*

In [358]:
variable_objetivo = "temp_max_manana"

## 2. Exploración de los datos 


### 2.1 Estructura general del dataset


*Se revisaron las dimensiones, los tipos, las primeras filas y la información general del dataset.*

In [359]:
print("Dimensiones del dataset:")
print(f"Filas: {datos_entrenamiento.shape[0]}, Columnas: {datos_entrenamiento.shape[1]}")

print("\nPrimeras cinco filas:")
display(datos_entrenamiento.head())

Dimensiones del dataset:
Filas: 2576, Columnas: 27

Primeras cinco filas:


,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
0,2009-01-01,999.1456,996.50,1000.87,1.3993,0.910860,0.875000,94.8,1.7650,0.7786,...,-0.3618,-0.0223,183.5308,143.0,2009.0,1.0,invierno,January,S,-2.12
1,2009-01-02,999.6006,997.93,1002.65,1.5039,0.920868,86.600000,96.3,2.7588,1.4195,...,0.4267,0.3689,40.8436,144.0,2009.0,2.0,invierno,JULY,NE,-0.82
2,2009-01-03,998.5486,993.05,1002.49,3.1304,76.458100,48.390000,93.9,15.1796,1.2509,...,-0.6993,-0.5268,216.9916,144.0,2009.0,3.0,invierno,January,SO,-0.63
3,2009-01-04,988.5107,985.12,992.93,2.3223,89.417400,97.946704,NaN,4.4904,1.7204,...,-1.1268,-1.0413,222.7419,144.0,2009.0,4.0,invierno,JANUARY,SO,-1.44
4,2009-01-05,990.4057,NaN,997.54,4.2315,86.260400,74.600000,93.2,5.3922,3.8003,...,2.6275,0.2874,6.2418,NaN,2009.0,5.0,East,January,N,-10.88


*No se incluyeron nuevamente las funciones `dtypes` e `info` porque los tipos de variables y la información general ya fueron mostrados en el diccionario de datos. De esta manera, se evitó repetir información y se conservaron en esta sección únicamente las revisiones adicionales de estructura.*

### 2.2 Estadísticas descriptivas


*Se resumieron las variables con estadísticas descriptivas apropiadas.*

In [360]:
print("Estadísticas descriptivas:")
display(datos_entrenamiento.describe(include="all").T)

Estadísticas descriptivas:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
fecha,2504,2486,2009-01-22,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
presion_media,2501.0,NaN,NaN,NaN,1010.494271,435.334494,948.996,984.2878,989.4528,994.469,9939.918
presion_min,2502.0,NaN,NaN,NaN,986.684409,8.927138,913.6,981.371333,987.07,992.5675,1012.320198
presion_max,2512.0,NaN,NaN,NaN,991.704244,7.726906,951.94,987.07,991.935,996.7825,1013.94
presion_desv,2496.0,NaN,NaN,NaN,1.752041,1.215931,0.2269,0.89905,1.44455,2.2548,12.3975
humedad_media,2502.0,NaN,NaN,NaN,52.413586,35.975384,0.004663,0.86906,68.5581,80.5626,100.0
humedad_min,2496.0,NaN,NaN,NaN,45.094067,30.196163,0.2377,1.01179,49.67,67.665,103.311405
humedad_max,2490.0,NaN,NaN,NaN,90.236582,10.647237,27.02,88.025,93.3,96.6,100.0
humedad_desv,2515.0,NaN,NaN,NaN,10.606821,5.481338,0.0,6.0821,10.1998,14.7222,26.3416
viento_media,2490.0,NaN,NaN,NaN,3.814103,3.307402,0.0,1.65535,2.4634,4.880505,24.7536


*Los valores `25%`, `50%` y `75%` corresponden a los cuartiles de las variables numéricas. El primer cuartil (`25%`) indica que aproximadamente el 25% de los datos tiene un valor menor o igual a ese punto. El segundo cuartil (`50%`) es la mediana, es decir, divide los datos en dos partes iguales. El tercer cuartil (`75%`) indica que aproximadamente el 75% de los datos tiene un valor menor o igual a ese punto. En las variables categóricas, `freq` representa cuántas veces aparece la categoría más frecuente, no un porcentaje.*

## 3. Calidad de los datos

| Dimensión | Pregunta que responde |
|---|---|
| Completitud | ¿Faltan datos? |
| Unicidad | ¿Hay registros duplicados? |
| Consistencia | ¿Los valores del mismo concepto están representados de forma uniforme? |
| Validez | ¿Los valores están dentro del rango permitido y tienen sentido en el contexto? |

### 3.1 Completitud

*Se revisó la presencia de datos faltantes en las variables del conjunto.*

In [361]:
((data.isnull().sum() / data.shape[0])).sort_values(ascending=False)

temp_max_manana      0.036879
viento_min           0.035326
mes                  0.034161
estacion_anio        0.034161
anio                 0.033385
humedad_max          0.033385
viento_media         0.033385
viento_max           0.031444
presion_desv         0.031056
humedad_min          0.031056
viento_norte         0.031056
rafaga_desv          0.030668
direccion_viento     0.030668
rafaga_media         0.030280
registros_del_dia    0.029891
presion_media        0.029115
humedad_media        0.028727
presion_min          0.028727
rafaga_max           0.028339
rafaga_min           0.027950
viento_desv          0.027950
fecha                0.027950
sector_viento        0.027562
dia_del_anio         0.026398
viento_este          0.024845
presion_max          0.024845
humedad_desv         0.023680
dtype: float64

*Se observó que todas las variables del conjunto de datos presentan valores faltantes. Para cada variable se informó su nombre, el número de filas con valores nulos y los primeros índices correspondientes, sin imprimir tablas completas.*

In [362]:
for variable in data.columns:
    nulos = data[data[variable].isna()].copy()
    indices = data.index[data[variable].isna()]

    print(f"\n{'=' * 70}")
    print(f"Variable: {variable}")
    print(f"Filas con {variable} nula: {nulos.shape[0]}")
    print(f"Primeros índices de los registros nulos: {indices[:10].tolist()}")


Variable: fecha
Filas con fecha nula: 72
Primeros índices de los registros nulos: [85, 96, 224, 237, 266, 323, 403, 475, 537, 660]

Variable: presion_media
Filas con presion_media nula: 75
Primeros índices de los registros nulos: [72, 165, 167, 174, 195, 201, 217, 230, 256, 308]

Variable: presion_min
Filas con presion_min nula: 74
Primeros índices de los registros nulos: [4, 5, 7, 81, 133, 143, 185, 233, 235, 260]

Variable: presion_max
Filas con presion_max nula: 64
Primeros índices de los registros nulos: [10, 102, 112, 115, 137, 149, 216, 280, 336, 363]

Variable: presion_desv
Filas con presion_desv nula: 80
Primeros índices de los registros nulos: [23, 43, 45, 46, 66, 101, 116, 139, 178, 181]

Variable: humedad_media
Filas con humedad_media nula: 74
Primeros índices de los registros nulos: [31, 80, 116, 169, 206, 235, 279, 284, 298, 305]

Variable: humedad_min
Filas con humedad_min nula: 80
Primeros índices de los registros nulos: [26, 55, 56, 248, 264, 337, 403, 421, 427, 481]



### 3.2 Unicidad

*Se revisó la existencia de registros duplicados en el conjunto.*

In [363]:
duplicados = data[data.duplicated(keep=False)].copy()

print(f"Total de filas duplicadas: {duplicados.shape[0]}")
print("Muestra de los primeros 10 registros duplicados:")
display(duplicados.head(10))

Total de filas duplicadas: 8
Muestra de los primeros 10 registros duplicados:


,fecha,presion_media,presion_min,presion_max,presion_desv,humedad_media,humedad_min,humedad_max,humedad_desv,viento_media,...,viento_norte,viento_este,direccion_viento,registros_del_dia,anio,dia_del_anio,estacion_anio,mes,sector_viento,temp_max_manana
21,2009-01-22,9784.1150,971.48,983.54,3.9853,86.731100,69.11,97.6,9.7926,10.09692,...,-2.7229,-0.1899,183.9897,144.0,2009.0,22.0,verano,octubre,S,5.770
1412,2012-11-13,1005.3724,1002.77,1006.81,1.0690,88.175500,58.28,99.1,12.6853,4.07916,...,-0.9230,-0.2193,193.3681,144.0,2012.0,318.0,otono,abril,S,44.852
1499,2013-02-08,984.6369,983.06,987.49,1.3454,0.886056,0.76,96.8,4.6160,NaN,...,0.0254,-0.2613,275.5529,144.0,2013.0,39.0,verano,Sep,O,-0.600
1570,2013-04-20,1000.4455,998.46,1002.49,1.1632,60.748100,45.71,82.0,10.9518,17.65908,...,4.1573,2.5121,31.1426,144.0,2013.0,110.0,primavera,JANUARY,NE,61.412
2556,2013-04-20,1000.4455,998.46,1002.49,1.1632,60.748100,45.71,82.0,10.9518,17.65908,...,4.1573,2.5121,31.1426,144.0,2013.0,110.0,primavera,JANUARY,NE,61.412
2559,2012-11-13,1005.3724,1002.77,1006.81,1.0690,88.175500,58.28,99.1,12.6853,4.07916,...,-0.9230,-0.2193,193.3681,144.0,2012.0,318.0,otono,abril,S,44.852
2563,2009-01-22,9784.1150,971.48,983.54,3.9853,86.731100,69.11,97.6,9.7926,10.09692,...,-2.7229,-0.1899,183.9897,144.0,2009.0,22.0,verano,octubre,S,5.770
2565,2013-02-08,984.6369,983.06,987.49,1.3454,0.886056,0.76,96.8,4.6160,NaN,...,0.0254,-0.2613,275.5529,144.0,2013.0,39.0,verano,Sep,O,-0.600


### 3.3 Consistencia

*Se revisó que los valores de un mismo concepto estuvieran representados de forma uniforme.*

In [364]:
# TODO: analizar la consistencia de los valores.

### 3.4 Validez

*Se revisó que los valores estuvieran dentro del rango permitido y fueran pertinentes para el contexto.*

In [365]:
# TODO: analizar la validez de los valores.

## 4. Preparación de los datos

*Se prepararon los datos de forma reproducible y evitando fuga de información.*

### 4.1 Decisiones de limpieza (justificación basada en la sección 2)

*Se justificaron las decisiones de limpieza usando los hallazgos de la sección 2.*

In [366]:
# TODO: documentar las decisiones de limpieza con base en la sección 2.


### 4.2 Tratamiento de duplicados

*Se revisó la existencia de registros duplicados. Para evitar una salida extensa, se mostró únicamente una muestra de los primeros 10 registros duplicados y se informó el total encontrado.*

In [367]:
# TODO: tratar los registros duplicados según la estrategia definida.

### 4.3 Ingeniería de características

*Se construyeron características justificadas por el problema y los datos disponibles.*

In [368]:
# TODO: crear las características derivadas pertinentes.


### 4.4 Codificación y escalamiento

*Se codificaron las variables categóricas y se escalaron las variables numéricas cuando correspondía.*

In [369]:
# TODO: definir la codificación y el escalamiento dentro de los pipelines.


### 4.5 División entrenamiento / test (test_size=0.25, random_state=42)

*Se dividieron los datos con `test_size=0.25` y `random_state=42`.*

In [370]:
# TODO: dividir los datos con test_size=0.25 y random_state=42.


### 4.6 Justificación de las decisiones tomadas (celda Markdown)

*Se explicaron las decisiones de preparación y sus implicaciones para el modelo.*

**Justificación:**

## 5. Modelamiento – Regresión lineal

*Se construyeron y compararon dos pipelines de regresión lineal.*

### 5.1 Modelo 1: pipeline con estrategia de preparación A

*Se definió el primer pipeline y se describió su estrategia de preparación A.*

In [371]:
# TODO: implementar el pipeline con estrategia de preparación A.


### 5.2 Modelo 2: pipeline con estrategia de preparación B

*Se definió el segundo pipeline y se describió su estrategia de preparación B.*

In [372]:
# TODO: implementar el pipeline con estrategia de preparación B.


### 5.3 Entrenamiento y validación (métrica base: RMSE)

*Se entrenaron y validaron ambos modelos usando RMSE como métrica base.*

In [373]:
# TODO: entrenar y validar ambos modelos usando RMSE.


## 6. Evaluación cuantitativa

*Se evaluaron cuantitativamente los modelos sobre datos no vistos.*

### 6.1 Cálculo de métricas sobre test (RMSE, MAE, R²)

*Se calcularon RMSE, MAE y R² sobre el conjunto de test.*

In [374]:
# TODO: calcular RMSE, MAE y R² sobre test.


### 6.2 Tabla comparativa de los dos mejores modelos

*Se presentó una tabla comparable con las métricas de los dos modelos.*

In [375]:
# TODO: construir la tabla comparativa de los dos modelos.


### 6.3 Selección del mejor modelo (justificación)

*Se seleccionó el mejor modelo y se justificó la decisión con las métricas.*

## 7. Evaluación cualitativa e interpretación

*Se interpretó el mejor modelo y se revisaron sus supuestos cualitativamente.*

### 7.1 Coeficientes del mejor modelo

*Se interpretaron el intercepto y los coeficientes del mejor modelo.*

In [376]:
# TODO: extraer y analizar los coeficientes del mejor modelo.


### 7.2 Tabla de importancia de variables

*Se organizaron los coeficientes o medidas disponibles en una tabla de importancia.*

In [377]:
# TODO: construir la tabla de importancia de variables.


### 7.3 Validación de los supuestos de la regresión lineal

*Se revisaron la linealidad, la independencia, la homocedasticidad y la normalidad de los residuos cuando fue posible.*

In [378]:
# TODO: validar los supuestos de la regresión lineal.


## 8. Análisis de resultados

*Se respondieron las preguntas con evidencia obtenida del análisis, sin inventar resultados.*

### 8.1 ¿Cuál fue el valor de los diferentes coeficientes obtenidos en el mejor modelo?

*Se respondió con los valores calculados y su interpretación.*

**Respuesta:**

### 8.2 A partir de la tabla comparativa, ¿cuál modelo ofrece el mejor rendimiento sobre test? ¿Qué interpretación tienen las métricas?

*Se respondió comparando RMSE, MAE y R², e interpretando sus magnitudes.*

**Respuesta:**

### 8.3 ¿Cuáles variables fueron seleccionadas? ¿Qué interpretación tienen frente al problema y cómo apoyan la toma de decisiones?

*Se describieron las variables seleccionadas, su interpretación y utilidad para decidir.*

**Respuesta:**

### 8.4 ¿Cómo se representa matemáticamente la regresión lineal en este contexto? Método utilizado y proceso de solución.

*Se explicó la representación matemática, el método utilizado y el proceso de solución.*

**Respuesta:**

### 8.5 ¿Qué tipos de sesgo podrían afectar los resultados? Describir dos.

*Se describieron dos sesgos posibles y cómo podrían afectar las conclusiones.*

**Respuesta:**

## 9. Uso del modelo: predicciones sobre datos no etiquetados

*Se aplicó el mejor modelo al archivo de test no etiquetado y se documentó la salida.*

### 9.1 Aplicación del mejor modelo al archivo de test

*Se generaron predicciones para el archivo de test no etiquetado.*

In [379]:
# TODO: aplicar el mejor modelo al archivo de test.


### 9.2 Exportación de resultados a "Datos test Lab1.csv"

*Se exportaron las predicciones al archivo "Datos test Lab1.csv".*

In [380]:
# TODO: exportar resultados a "Datos test Lab1.csv".


## 10. Conclusiones

*Se resumieron los resultados, las decisiones, las limitaciones y los aprendizajes del laboratorio.*

**Conclusiones:**

## 11. Uso de herramientas de IA generativa

*Se declaró y analizó críticamente el uso de herramientas de IA generativa.*

### 11.1 Declaración del uso (herramienta y tipo de uso)

*Se indicó qué herramienta se usó y para qué tipo de apoyo.*

**Respuesta:**

### 11.2 Prompts utilizados

*Se registraron los prompts utilizados durante el trabajo.*

**Prompts:**

### 11.3 Análisis crítico del resultado (responder al menos dos de las preguntas guía del curso)

*Se respondieron al menos dos preguntas guía y se evaluó críticamente el resultado de la IA.*

**Respuesta:**

### 11.4 Aportes propios de los estudiantes

*Se describieron los aportes, las decisiones y las verificaciones realizadas por los estudiantes.*

**Aportes:**

## 12. Referencias

*Se registraron las fuentes de datos, la documentación y la bibliografía utilizadas.*

**Referencias:**